# Camera Manual RAG System

This notebook presents a GitHub-friendly portfolio version of the Camera Manual RAG System. The original assignment notebook was developed in Google Colab and used four Olympus / OM System camera manuals as the document corpus.

## Objective

Build a retrieval-augmented generation system that can answer natural-language questions about technical camera manuals, such as how to enable silent shooting, use Face Priority AF, connect Wi-Fi, format memory cards, or locate ISO/self-timer settings.

## Pipeline

1. Parse PDF manuals with high-resolution PDF partitioning.
2. Extract text and OCR image-based content.
3. Apply semantic sentence-based chunking.
4. Embed chunks using `intfloat/e5-base-v2`.
5. Store embeddings in ChromaDB.
6. Combine BM25 and vector retrieval.
7. Improve retrieval with HyDE-style hypothetical embeddings.
8. Rerank retrieved contexts with Cohere Rerank.
9. Generate answers with Gemini 2.0 Flash.
10. Evaluate answers with RAGAS.

In [ ]:
import os
import getpass
import pickle
import shutil
import time

import cohere
import google.generativeai as genai
import nltk
import pytesseract
from PIL import Image
from nltk.tokenize import sent_tokenize
from langchain.schema import Document
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from unstructured.partition.pdf import partition_pdf

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY') or getpass.getpass('Enter your Gemini API key: ')
genai.configure(api_key=GEMINI_API_KEY)

In [ ]:
def semantic_chunking(documents, max_words=100):
    chunks = []
    for doc in documents:
        sentences = sent_tokenize(doc.page_content)
        buffer, count = [], 0
        for sentence in sentences:
            buffer.append(sentence)
            count += len(sentence.split())
            if count >= max_words:
                chunks.append(Document(page_content=' '.join(buffer), metadata=doc.metadata))
                buffer, count = [], 0
        if buffer:
            chunks.append(Document(page_content=' '.join(buffer), metadata=doc.metadata))
    return chunks

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name='intfloat/e5-base-v2',
    encode_kwargs={'normalize_embeddings': True}
)

# vectorstore = Chroma.from_documents(
#     documents=all_chunks,
#     embedding=embedding_model,
#     persist_directory='olympus_vector_db',
#     collection_name='olympus_manual'
# )

In [ ]:
def normalize_query(query):
    return query.lower().strip()

def get_hypothetical_embedding(query, llm, embed_model):
    prompt = f'Generate a plausible manual-like answer to the question: {query}'
    hypothetical_answer = llm.generate_content(prompt).text
    return embed_model.embed_query(hypothetical_answer)

def rerank_documents(query, documents, cohere_client):
    passages = [doc.page_content for doc in documents]
    response = cohere_client.rerank(
        query=query, documents=passages, model='rerank-english-v3.0'
    )
    return [documents[item.index] for item in response.results]

## Notes

The full original Colab notebook and generated vector database are kept locally. Raw manuals and generated Chroma database files are excluded from GitHub by default because of licensing and file-size considerations.